In [ ]:
import os
import sys

os.chdir("/Users/morizin/Documents/Code/crash-detection-project")
sys.path.append("src")

In [ ]:
from crash_detection.config import ConfigurationManager

config = ConfigurationManager(config_path="config/config.yaml", latest=True)

In [ ]:
from torch.onnx import export
from crash_detection.utils.common import load_pickle
from crash_detection.config.artifact_entity import ModelTrainingArtifact

model_training_artifact = ModelTrainingArtifact(**load_pickle(path = config.artifact_path / "models" / "video-classifier1"/ "model_training_artifact.pkl"))

In [ ]:
from crash_detection.components.model.model import Model

model = Model.from_pretrained(model_path=model_training_artifact.model_path)
model.eval()

In [ ]:
import torch 

dummy_input =  torch.randn(2, 20, 224, 224).to("mps:0")

torch.onnx.export(
    model=model,
    f = config.artifact_path // "onnx" // "video-classifier1" / "model.onnx",
    args=dummy_input,
    opset_version=18,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    external_data=False
)

In [ ]:
import torch
import numpy as np
import onnxruntime as ort
from scipy.special import expit

dummy_input =  torch.randn(20, 20, 224, 224).to("cpu")

ort_session = ort.InferenceSession(str(config.artifact_path // "onnx" // "video-classifier1" / "model.onnx"))

ort_inputs = {ort_session.get_inputs()[0].name: np.zeros(shape=(20, 20, 224, 224)).astype(np.float32)}
ort_outs = ort_session.run(None, ort_inputs)

ort_outs = expit(ort_outs[0])
ort_outs

In [ ]:
from pydantic import BaseModel
from crash_detection.core import Directory
from typing import Optional
from pathlib import Path

class ModelExportingConfig(BaseModel):
    name: str
    model_path: Path | str
    onnx_opset: int
    outdir : Directory

class ModelExportingArtifact(BaseModel):
    model_path: Path
    onnx_model_path: Path
    input_shape: Optional[tuple] = None
    output_shape: Optional[tuple] = None

In [ ]:
from crash_detection.config.artifact_entity import ModelTrainingArtifact

class ConfigurationManager(ConfigurationManager):
    
    def get_model_exporting_config(self, model_training_artifact : ModelTrainingArtifact) -> ModelExportingConfig:
        return ModelExportingConfig(
            name = model_training_artifact.name,
            model_path = model_training_artifact.model_path,
            onnx_opset = self.config.models[model_training_artifact.name].onnx_opset,
            outdir = self.artifact_path // "onnx" // model_training_artifact.name
        )

In [ ]:
from crash_detection import logger
from torch.export import Dim

class ModelExportingComponent:
    def __init__(self, model_export_config : ModelExportingConfig):
        self.config = model_export_config
        self.model = Model.from_pretrained(model_path=model_export_config.model_path)
        self.model.eval()
        self.export_path = model_export_config.outdir

    def export_to_onnx(self,):
        dummy_input =  torch.randn(2, 20, 224, 224).to(self.model.device)
        return torch.onnx.export(
            model=self.model,
            f=self.export_path / "model.onnx",
            args=dummy_input,
            opset_version=self.config.onnx_opset,
            input_names=['inputs'],
            dynamic_shapes={
                'inputs': {0: "batch_size"}
            },
            external_data=False,
            dynamo = True,
            artifacts_dir=str(self.export_path),
            report=True,
            verify=True,
            # profile=True
        )

    def __call__(self, ):
        onnx_model = self.export_to_onnx()
        logger.info(f"ONNX model exported at : {self.export_path / 'model.onnx'}")
        print(onnx_model)
        
        return ModelExportingArtifact(
            name = self.config.name,
            model_path= self.config.model_path,
            onnx_model_path = self.export_path / "model.onnx",
            input_shape = None, output_shape = None
        )

In [ ]:
model_training_artifact = ModelTrainingArtifact(**load_pickle(path = config.artifact_path / "models" / "video-classifier1"/ "model_training_artifact.pkl"))

In [ ]:
config_manager = ConfigurationManager(config_path="config/config.yaml", latest=True)
model_exporting_config = config_manager.get_model_exporting_config(model_training_artifact=model_training_artifact)

In [ ]:
model_exporting_component = ModelExportingComponent(model_export_config=model_exporting_config)
model_exporting_artifact = model_exporting_component()

In [ ]:
import torch
import numpy as np
import onnxruntime as ort
from scipy.special import expit

dummy_input =  torch.randn(20, 20, 224, 224).to("cpu")

ort_session = ort.InferenceSession(str(config.artifact_path // "onnx" // "video-classifier1" / "model.onnx"))

ort_inputs = {ort_session.get_inputs()[0].name: np.zeros(shape=(20, 20, 224, 224)).astype(np.float32)}
ort_outs = ort_session.run(None, ort_inputs)

ort_outs = expit(ort_outs[0])
ort_outs